# 111. DS-Net from the authors' code, more settings and seeds

DS-Net (Li et al., CVPR 2021 oral) Stage I as released, from the
authors' repository with `third_party/patches/dsnet.patch`, exactly as
`dsnet_ib` of notebook 102: the sandwich ['largest', 'uniform',
'uniform', 'smallest'], soft targets from an EMA of the supernet at its
largest width (decay 0.997), ensemble IB off, read at the sixteen widths
after BN recalibration on 20 batches. Our recipe in every setting: 100
epochs, batch 256, SGD nesterov lr 0.2, 5 warm-up epochs then cosine, wd
5e-4, clip 5. The models are built from their slimmable ops with the
channel counts of our US-Net rows at all sixteen widths (checked layer by
layer, `dsnet_README.md`).

| | GPU | setting | seed |
|---|---|---|---|
| dsnet_ib_mbv2_slimlast_s2026 | 0 | CIFAR MobileNetV2 with slim_last | 2026 |
| dsnet_ib_mbv2_slimlast_s2006 | 1 | CIFAR MobileNetV2 with slim_last | 2006 |

Notebooks 102 and 108 to 111 together give three seeds in each of the
three settings (R50 seed 1995 is `dsnet_ib` of notebook 102).

## More than one session

CIFAR MobileNetV2 with slim_last: about 6 to 8 hours on a T4.
Each run writes `ckpt/<label>/last.pt` after every epoch.
When the session ends, Save Version, then in a new run attach that
version's output (pick the version by number, not Latest) and run again:
the notebook finds the checkpoints and the runs continue. Attach the
CIFAR-100 dataset as well; Tiny ImageNet is found under /kaggle/input if
attached, or downloaded.

In [ ]:
# Fixed for this notebook.
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'
# key: (repository, pinned commit, folder, patch, extra pip packages)
UPSTREAMS = {'dsnet': ('https://github.com/changlin31/DS-Net.git', '15cd3036970ec27d2c306014344fd50d9e9b888b', 'DS-Net', 'dsnet.patch', 'timm')}
WORKER_ENV = {'DSNET_WORKERS': '2'}

In [ ]:
import os, queue, re, shutil, subprocess, sys, threading, time
import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
if n_gpu < 2:
    raise SystemExit('Needs GPU T4 x2: one card per method.')

WORK = '/kaggle/working'
OURS = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(OURS):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, OURS],
                   check=True)
print('our code at', subprocess.run(
    ['git', '-C', OURS, 'log', '-1', '--format=%h %s'],
    capture_output=True, text=True).stdout.strip())

CODE = {}
packages = []
for key, (url, commit, folder, patch, pip) in UPSTREAMS.items():
    CODE[key] = os.path.join(WORK, folder)
    if not os.path.isdir(CODE[key]):
        subprocess.run(['git', 'clone', url, CODE[key]], check=True)
        subprocess.run(['git', '-C', CODE[key], 'checkout', '-q', commit],
                       check=True)
        subprocess.run(['git', '-C', CODE[key], 'apply', os.path.join(
            OURS, 'third_party', 'patches', patch)], check=True)
    print(key, 'at', subprocess.run(
        ['git', '-C', CODE[key], 'log', '-1', '--format=%h %ad',
         '--date=short'], capture_output=True, text=True).stdout.strip(),
        '+', patch)
    packages += pip.split()

# install without letting pip replace Kaggle's torch
base = torch.__version__.split('+')[0]
with open(os.path.join(WORK, 'constraints.txt'), 'w') as handle:
    handle.write('torch=={}\n'.format(base))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c',
                os.path.join(WORK, 'constraints.txt')] + packages, check=True)

In [ ]:
# the folder that contains cifar-100-python/, or a place to download it
DATA = None
for root, dirs, _ in os.walk('/kaggle/input', followlinks=True):
    if 'cifar-100-python' in dirs:
        DATA = root
        break
if DATA is None:
    DATA = os.path.join(WORK, 'data')
    os.makedirs(DATA, exist_ok=True)
    print('no CIFAR-100 attached: it will be downloaded into', DATA)
else:
    print('CIFAR-100 from', DATA)

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs):
    """one command per card, output tagged and interleaved"""
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for label, gpu, cwd, command in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, **WORKER_ENV)
        proc = subprocess.Popen(command, cwd=cwd, env=env, text=True,
                                stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] gpu {}: {}'.format(label, gpu, ' '.join(command)),
              flush=True)
    started, remaining = time.time(), len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[round(float(width), 2)] = (
                float(loss), float(top1))
        print('[{}] {}'.format(label, line), flush=True)
    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
CK = os.path.join(WORK, 'ckpt')
# DS-Net keeps ckpt/<label>/last.pt
wanted = {'dsnet_ib_mbv2_slimlast_s2026': ('last.pt', 'dsnet_ib_mbv2_slimlast_s2026'),
          'dsnet_ib_mbv2_slimlast_s2006': ('last.pt', 'dsnet_ib_mbv2_slimlast_s2006')}
for label, (name, folder) in wanted.items():
    target = os.path.join(CK, folder or '', name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        for root, _, files in os.walk('/kaggle/input', followlinks=True):
            if name in files and (folder is None
                                  or os.path.basename(root) == folder):
                shutil.copy(os.path.join(root, name), target)
                print('restored', label, 'from', root)
                break
    print(label, 'resumes from its checkpoint' if os.path.exists(target)
          else 'starts at epoch 1')

In [ ]:
# each run ends with the sixteen protocol lines
codes = run_pinned([
    ('dsnet_ib_mbv2_slimlast_s2026', '0', CODE['dsnet'],
     [sys.executable, '-u', 'train.py', '-c', 'configs/cifar100_ds_mobilenetv2_slimlast_uniform.yml',
      '--data_dir', DATA, '--seed', '2026', '--skip_upstream_test',
      '--output', os.path.join(WORK, 'dsnet_out', 'dsnet_ib_mbv2_slimlast_s2026'),
      '--ckpt_dir', os.path.join(CK, 'dsnet_ib_mbv2_slimlast_s2026'), '--log_prefix', 'dsnet_ib_mbv2_slimlast_s2026']),
    ('dsnet_ib_mbv2_slimlast_s2006', '1', CODE['dsnet'],
     [sys.executable, '-u', 'train.py', '-c', 'configs/cifar100_ds_mobilenetv2_slimlast_uniform.yml',
      '--data_dir', DATA, '--seed', '2006', '--skip_upstream_test',
      '--output', os.path.join(WORK, 'dsnet_out', 'dsnet_ib_mbv2_slimlast_s2006'),
      '--ckpt_dir', os.path.join(CK, 'dsnet_ib_mbv2_slimlast_s2006'), '--log_prefix', 'dsnet_ib_mbv2_slimlast_s2006'])])
print(codes)

## Results, against A (BX, 76.86) and K (DD, 77.50)

Send back this table and the `val ... -1/100` lines above it.

The A and K columns are the ResNet-50 CIFAR-100 references: they mean
something only for the R50 rows. Read the other rows against the US-Net
and SlimOT rows of their own setting.

In [ ]:
A_REF = {0.25: 75.26, 0.3: 75.35, 0.35: 75.54, 0.4: 76.01, 0.45: 76.3, 0.5: 76.77, 0.55: 76.63, 0.6: 77.03, 0.65: 77.13, 0.7: 77.2, 0.75: 77.39, 0.8: 77.56, 0.85: 77.6, 0.9: 77.99, 0.95: 78.01, 1.0: 78.04}
K_REF = {0.25: 75.63, 0.3: 76.29, 0.35: 76.54, 0.4: 76.97, 0.45: 77.67, 0.5: 77.43, 0.55: 77.58, 0.6: 77.62, 0.65: 77.83, 0.7: 78.02, 0.75: 78.0, 0.8: 78.01, 0.85: 78.09, 0.9: 78.23, 0.95: 78.08, 1.0: 77.95}
labels = ['dsnet_ib_mbv2_slimlast_s2026', 'dsnet_ib_mbv2_slimlast_s2006']
print('{:>7}{:>8}{:>8}'.format('width', 'A', 'K') + ''.join(
    '{:>12}{:>7}{:>7}'.format(b[:11], 'vs A', 'vs K') for b in labels))
for width in sorted(A_REF):
    row = '{:>7.2f}{:>8.2f}{:>8.2f}'.format(
        width, A_REF[width], K_REF[width])
    for label in labels:
        entry = results.get(label, {}).get(width)
        if entry is None:
            row += '{:>12}{:>7}{:>7}'.format('-', '-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>12.2f}{:>+7.2f}{:>+7.2f}'.format(
            accuracy, accuracy - A_REF[width], accuracy - K_REF[width])
    print(row)
reference = sum(A_REF.values()) / len(A_REF)
k_reference = sum(K_REF.values()) / len(K_REF)
print('\n{:24} mean {:.2f}'.format('A (BX)', reference))
print('{:24} mean {:.2f}'.format('K (DD)', k_reference))
for label in labels:
    table = results.get(label, {})
    if len(table) == len(A_REF):
        mean = sum(100.0 * (1.0 - v[1]) for v in table.values()) / len(table)
        print('{:24} mean {:.2f}   vs A {:+.2f}   vs K {:+.2f}'.format(
            label, mean, mean - reference, mean - k_reference))
    else:
        print('{:24} {} of 16 widths read'.format(label, len(table)))

In [ ]:
# keep the checkpoints in the output, under a name the next session finds
print('checkpoints in', CK, sorted(os.listdir(CK)))